# RecSys 2026 — Devset Two-Step Inference (Colab)

Runs the full 8000-row devset inference for a two-step (retrieval + LLM-response) experiment on Colab GPU, then zips the prediction JSON for download.

## Setup checklist

1. **Runtime → Change runtime type → T4 / L4 / A100 GPU**. T4 is enough for Qwen 1.5B at bf16.
2. Run all cells top-to-bottom.
3. When inference finishes, the prediction zip is at `/content/020-two-step-wrrf-lyrics-qwen15b-devset.zip`. Download via the browser popup (cell 7a) or the file browser (📁 sidebar).

Wall time: ~15 min on T4, ~5 min on A100. Output JSON is ~15 MB; zipped ~3 MB.

**Drive is NOT required** — repo is cloned fresh from GitHub each run; only mount Drive (cell 7b) if you want the JSON to survive Colab disconnects.

## What this runs

- **Config**: `music-crs-baselines/config/020-two-step-wrrf-lyrics-qwen15b-devset.yaml`
- **Retrieval**: wRRF(BM25 4-field + dense-metadata-qwen3 + dense-lyrics-qwen3) — Wave 1 best retrieval branch (nDCG@20=0.0994)
- **Response LM**: Qwen/Qwen2.5-1.5B-Instruct at bf16 on CUDA
- **Batch size**: 64 (T4 headroom; bump to 128 on A100 for faster run)

Branch: **`fresh-model`** (change `BRANCH` in cell 2 if needed).

In [ ]:
# 1) Verify GPU.
!nvidia-smi | head -20

In [ ]:
# 2) FORCE-FRESH clone — delete any prior copy first so re-runs in the same
# Colab runtime always pull the latest code from GitHub.
BRANCH = 'fresh-model'
!rm -rf /content/recsys2026-lora-tutorial
!git clone -b {BRANCH} https://github.com/orrimoch/recsys2026-lora-tutorial.git /content/recsys2026-lora-tutorial
%cd /content/recsys2026-lora-tutorial

# Sanity check: print the commit you're about to run on. Compare against the
# top commit at https://github.com/orrimoch/recsys2026-lora-tutorial/commits/fresh-model
print('\n=== CODE VERSION CHECK ===')
!git log -1 --pretty=format:'commit:  %h%nauthor:  %an%ndate:    %ai%nsubject: %s'
print()
!echo -n 'branch:  ' && git rev-parse --abbrev-ref HEAD

In [ ]:
# 3) Install pinned deps. Colab usually has compatible torch — pip resolves the rest.
!pip install -q -r requirements.txt
!python -c "import torch, transformers, bm25s; print('torch', torch.__version__, 'cuda', torch.cuda.is_available(), 'transformers', transformers.__version__, 'bm25s', bm25s.__version__)"

In [ ]:
# 4) (Optional) HuggingFace token — faster downloads, no anonymous rate limit.
# import os; os.environ['HF_TOKEN'] = 'hf_...'  # uncomment + paste your token

In [ ]:
# 5) Run the devset two-step inference.
#    --device cuda     : overrides the yaml's 'mps' (local M4 default)
#    --batch_size 64   : T4-safe for Qwen 1.5B + 2 query-encoder instances
#
# First run also builds dense-retrieval track-matrix caches (~2 min) and
# downloads 4 HF datasets (~2 min). Subsequent runs in the same Colab
# session reuse them.
TID = '020-two-step-wrrf-lyrics-qwen15b-devset'
BATCH_SIZE = 64
!cd music-crs-baselines && python run_inference_devset.py \
    --tid {TID} \
    --batch_size {BATCH_SIZE} \
    --device cuda

In [ ]:
# 6) Zip the prediction JSON for download.
import json, os, shutil
TID = '020-two-step-wrrf-lyrics-qwen15b-devset'
SRC = f'music-crs-baselines/exp/inference/devset/{TID}.json'
assert os.path.isfile(SRC), f'prediction not found at {SRC} — did inference fail?'

# Quick schema sanity before zipping.
with open(SRC) as f:
    rows = json.load(f)
print(f'rows: {len(rows)} (expected 8000 for full dev)')
assert len(rows) >= 8000, f'only {len(rows)} rows — partial run; do not ship'
sample_resp = rows[0].get('predicted_response', '')
print(f'sample response[0]: {sample_resp[:160]!r}')
assert sample_resp.strip(), 'first row has empty response — generation may have failed'

# Stage the file under a clean directory so shutil.make_archive zips just it.
stage = f'/content/_stage_{TID}'
shutil.rmtree(stage, ignore_errors=True)
os.makedirs(stage, exist_ok=True)
shutil.copy(SRC, os.path.join(stage, f'{TID}.json'))
zip_base = f'/content/{TID}'
shutil.make_archive(zip_base, 'zip', stage)
print('wrote', zip_base + '.zip')
!ls -lh {zip_base}.zip

In [ ]:
# 7a) Download via browser (no Drive needed).
from google.colab import files
TID = '020-two-step-wrrf-lyrics-qwen15b-devset'
files.download(f'/content/{TID}.zip')

In [ ]:
# 7b) OPTIONAL — mount Drive ONLY if you want the prediction to survive Colab disconnects.
# Uncomment to save a copy to Drive.
#
# from google.colab import drive
# drive.mount('/content/drive')
# import os, shutil
# TID = '020-two-step-wrrf-lyrics-qwen15b-devset'
# os.makedirs('/content/drive/MyDrive/recsys2026-predictions', exist_ok=True)
# shutil.copy(f'/content/{TID}.zip', '/content/drive/MyDrive/recsys2026-predictions/')
# print('saved to Drive')

## Done. On your local M4 / Linux:

```bash
cd recsys2026
TID=020-two-step-wrrf-lyrics-qwen15b-devset

# Drop the downloaded prediction into place.
unzip -o ~/Downloads/${TID}.zip -d music-crs-baselines/exp/inference/devset/

# Score + append row to submissions_log.md + run Wave 2 integration tests.
source recsys26/bin/activate
python scripts/run_experiment.py --tid ${TID} --skip-inference   # or: python scripts/local_eval.py ${TID}
pytest tests/test_wave2_integration.py -v
```

If Wave 2 integration tests go green, Wave 2 is closed.

## Re-using for other two-step experiments

Point the notebook at a different config by editing `TID` in cell 5 (and matching cell 6 / 7a / 8). The config file must live at `music-crs-baselines/config/{TID}.yaml` on the branch you cloned.